In [ ]:
!python -m pip install --upgrade pip
!python -m pip install --upgrade pandas openpyxl openai pycelonis_llm ipykernel

This cell checks the new documents which were submitted and their current status signature.

In [3]:
import pandas as pd

# Read the document status file, keeping every column as a string
doc_status_df = pd.read_excel("DocumentStatus.xlsx", dtype=str)

# Keep only the documents whose Status is null
null_status_df = doc_status_df.loc[
    doc_status_df["Status"].isna(), ["KYCAssessmentID", "DocumentNo", "CreationDate", "LastUpdated"]
].reset_index(drop=True)

print(f"Documents with null status: {len(null_status_df)} of {len(doc_status_df)}")
null_status_df

Documents with null status: 3 of 3


,KYCAssessmentID,DocumentNo,CreationDate,LastUpdated
0,921632,001,2026-11-10 00:00:00,NaN
1,927015,002,2026-11-10 00:00:00,NaN
2,954350,003,2026-11-10 00:00:00,NaN


This cell loads the PDF files which do not have a Status yet.

In [4]:
import base64
from pathlib import Path

DOCUMENTS_DIR = Path("Documents")

def read_pdf(document_no):
    """Return the raw bytes of Documents/<DocumentNo>.pdf, or None if it is missing."""
    pdf_path = DOCUMENTS_DIR / f"{document_no}.pdf"
    return pdf_path.read_bytes() if pdf_path.exists() else None

# Read the PDF for each document with a null status
null_status_df["PdfBytes"] = null_status_df["DocumentNo"].apply(read_pdf)
null_status_df["PdfBase64"] = null_status_df["PdfBytes"].apply(
    lambda b: base64.standard_b64encode(b).decode("utf-8") if b is not None else None
)

missing = null_status_df.loc[null_status_df["PdfBytes"].isna(), "DocumentNo"].tolist()
print(f"PDFs loaded: {len(null_status_df) - len(missing)} of {len(null_status_df)}")
if missing:
    print(f"Missing PDFs for DocumentNo: {missing}")
null_status_df[["KYCAssessmentID", "DocumentNo"]].assign(
    PdfSizeKB=null_status_df["PdfBytes"].apply(lambda b: round(len(b) / 1024, 1) if b else None)
)

PDFs loaded: 3 of 3


,KYCAssessmentID,DocumentNo,PdfSizeKB
0,921632,001,1117.9
1,927015,002,861.7
2,954350,003,195.1


This cell checks for digital signatures. If a digital signature is detected, then it updates the status to "Digital Signature Detected", and changes the LastUpdated status to the time of the runtime.

In [5]:
import re
from datetime import datetime

# A signed PDF contains a signature dictionary with a /ByteRange entry that lists the
# signed byte ranges. Empty (unsigned) signature fields have no /ByteRange.
BYTE_RANGE_PATTERN = re.compile(rb"/ByteRange\s*\[")

def has_digital_signature(pdf_bytes):
    """Return True if the PDF contains at least one applied digital signature."""
    return pdf_bytes is not None and BYTE_RANGE_PATTERN.search(pdf_bytes) is not None

null_status_df["HasDigitalSignature"] = null_status_df["PdfBytes"].apply(has_digital_signature)

# Timestamp for every status changed in this run, as a string to match dtype=str
last_updated = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Update the status of digitally signed documents
signed_mask = null_status_df["HasDigitalSignature"]
null_status_df["Status"] = None
null_status_df["LastUpdated"] = None
null_status_df.loc[signed_mask, "Status"] = "Digital Signature Detected"
null_status_df.loc[signed_mask, "LastUpdated"] = last_updated

# Carry the new status back to the full document status dataframe
signed_document_nos = null_status_df.loc[signed_mask, "DocumentNo"]
doc_status_df.loc[
    doc_status_df["DocumentNo"].isin(signed_document_nos), ["Status", "LastUpdated"]
] = ["Digital Signature Detected", last_updated]

print(f"Digital signatures detected: {signed_mask.sum()} of {len(null_status_df)}")
null_status_df[["KYCAssessmentID", "DocumentNo", "HasDigitalSignature", "Status", "LastUpdated"]]

Digital signatures detected: 0 of 3


,KYCAssessmentID,DocumentNo,HasDigitalSignature,Status,LastUpdated
0,921632,001,False,None,None
1,927015,002,False,None,None
2,954350,003,False,None,None


This section of code checks for the remaining documents which do not have a digital signature to see if there is a manual or no signature and updates the status of the document accordingly.

In [ ]:
import pycelonis_llm  # patches the supported LLM libraries to use the Celonis-provided LLMs
from pycelonis import get_celonis
from pycelonis_llm.llm import LLM

# --- Credentials ------------------------------------------------------------
# No API key is needed: inside the Celonis ML Workbench, PyCelonis takes its credentials
# from the environment, and the LLM calls go through Celonis.

# List the models available in this Celonis environment and pick GPT-5.6 Luna
available_models = [m.id for m in LLM(get_celonis().client).get_models()]
print("Available models:", available_models)

luna_models = [m for m in available_models if "gpt-5.6-luna" in m.lower()]
if not luna_models:
    raise ValueError("GPT-5.6 Luna is not available in this Celonis environment. Check the AI Settings / ML Workbench LLM.")
MODEL = luna_models[0]
print(f"Using model: {MODEL}")

In [ ]:
import openai
from pycelonis_llm.integrations.openai import CelonisOpenAI

# OpenAI client that authenticates with the Celonis environment's credentials
client = CelonisOpenAI()
print("Celonis OpenAI client ready")

In [ ]:
import json

# `client` and `MODEL` are set in the Celonis cells above
SIGNATURE_PROMPT = "Does this file contain a signature? Return with options Manual Signature and No Signature"
SIGNATURE_OPTIONS = ["Manual Signature", "No Signature"]

def classify_signature(pdf_base64, filename):
    """Send one PDF to GPT-5.6 Luna and return "Manual Signature" or "No Signature", or None on failure."""
    response = client.responses.create(
        model=MODEL,
        reasoning={"effort": "low"},
        text={
            "format": {
                "type": "json_schema",
                "name": "signature_result",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {"signature": {"type": "string", "enum": SIGNATURE_OPTIONS}},
                    "required": ["signature"],
                    "additionalProperties": False,
                },
            },
        },
        input=[{
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": filename,
                    "file_data": f"data:application/pdf;base64,{pdf_base64}",
                },
                {"type": "input_text", "text": SIGNATURE_PROMPT},
            ],
        }],
    )
    if response.status != "completed":
        reason = response.incomplete_details.reason if response.incomplete_details else None
        print(f"Unexpected response status '{response.status}' ({reason}, response {response.id})")
        return None
    if not response.output_text:
        print(f"No text returned, the model may have refused (response {response.id})")
        return None
    return json.loads(response.output_text)["signature"]

# Send the documents that still have a null status (and a loaded PDF) one at a time
to_classify = null_status_df.index[
    null_status_df["Status"].isna() & null_status_df["PdfBase64"].notna()
]
print(f"Documents to send to {MODEL}: {len(to_classify)}")

for idx in to_classify:
    document_no = null_status_df.at[idx, "DocumentNo"]
    try:
        signature = classify_signature(null_status_df.at[idx, "PdfBase64"], f"{document_no}.pdf")
    except openai.RateLimitError as e:
        print(f"{document_no}: rate limited, retry after {e.response.headers.get('retry-after')}s")
        continue
    except openai.APIStatusError as e:
        print(f"{document_no}: API error {e.status_code}: {e.message}")
        continue
    except openai.APIConnectionError:
        print(f"{document_no}: network error")
        continue
    if signature is None:
        continue

    # Update the status in both dataframes
    last_updated = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    null_status_df.loc[idx, ["Status", "LastUpdated"]] = [signature, last_updated]
    doc_status_df.loc[
        doc_status_df["DocumentNo"] == document_no, ["Status", "LastUpdated"]
    ] = [signature, last_updated]
    print(f"{document_no}: {signature}")

null_status_df[["KYCAssessmentID", "DocumentNo", "HasDigitalSignature", "Status", "LastUpdated"]]